# Titanic Survival Prediction

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.linear_model import LogisticRegression

## 1. Data Loading and Initial Exploration

In [2]:
# Load pandas DataFrame
df = pd.read_csv("titanic_data_updated.csv")

# to see what the data looks like
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
776,777,no,third,"Tobin, Mr. Roger",male,NaN,0,0,383121,7.7500,F38,Q
691,692,yes,third,"Karun, Miss. Manca",female,4.0,0,1,349256,13.4167,NaN,C
113,114,no,third,"Jussila, Miss. Katriina",female,20.0,1,0,4136,9.8250,NaN,S
300,301,yes,third,"Kelly, Miss. Anna Katherine ""Annie Kate""",female,NaN,0,0,9234,7.7500,NaN,Q
354,355,no,third,"Yousif, Mr. Wazli",male,NaN,0,0,2647,7.2250,NaN,C


In [3]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


## 2. Feature Engineering

In [4]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)
     

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
835,836,yes,first,"Compton, Miss. Sara Rebecca",female,39.0,1,1,PC 17756,83.1583,E49,C,3,E
777,778,yes,third,"Emanuel, Miss. Virginia Ethel",female,5.0,0,0,364516,12.4750,Missing,S,1,M
750,751,yes,second,"Wells, Miss. Joan",female,4.0,1,1,29103,23.0000,Missing,S,3,M
685,686,no,second,"Laroche, Mr. Joseph Philippe Lemercier",male,25.0,1,2,SC/Paris 2123,41.5792,Missing,C,4,M
311,312,yes,first,"Ryerson, Miss. Emily Borie",female,18.0,2,2,PC 17608,262.3750,B57 B59 B63 B66,C,5,B


In [5]:
df['Deck'].value_counts()

Deck
M    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [7]:
# Separate the features (X) from the target we want to predict (y)
# We drop 'Survived' from X because that's our answer key
X = df.drop('Survived', axis=1)

# y contains only the survival status
y = df['Survived']

In [8]:
X

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
0,1,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,Missing,S,2,M
1,2,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2,C
2,3,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,Missing,S,1,M
3,4,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,2,C
4,5,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,Missing,S,1,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,second,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,Missing,S,1,M
887,888,first,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,1,B
888,889,third,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,Missing,S,4,M
889,890,first,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,1,C


In [9]:
y

0       no
1      yes
2      yes
3      yes
4       no
      ... 
886     no
887    yes
888     no
889    yes
890     no
Name: Survived, Length: 891, dtype: object

## 3. Data Splitting

In [10]:
# Split data: 80% for training the model and 20% for testing its accuracy
# 'stratify=y' ensures both sets have a similar percentage of survivors
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 4. Outlier Handling and Data Cleaning

In [11]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
692,693,third,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,Missing,S,1,M
481,482,second,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,Missing,S,1,M
527,528,first,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,1,C
855,856,third,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,Missing,S,2,M
801,802,second,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,Missing,S,3,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,360,third,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,Missing,Q,1,M
258,259,first,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,Missing,C,1,M
736,737,third,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.0,1,3,W./C. 6608,34.3750,Missing,S,5,M
462,463,first,"Gee, Mr. Arthur H",male,47.0,0,0,111320,38.5000,E63,S,1,E


In [12]:
y_train

692    yes
481     no
527     no
855    yes
801    yes
      ... 
359    yes
258    yes
736     no
462     no
507    yes
Name: Survived, Length: 712, dtype: object

In [ ]:
#age
mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()

X_train['Z_score'] = (X_train['Age'] - mean_age) / std_age

musk = (abs(X_train['Z_score']) <= 3)

X_train = X_train[musk]
y_train = y_train[musk]

# fare

fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

IQR = fare_Q3 - fare_Q1

minimum = max(0 , fare_Q1 - 1.5 * IQR)
maximum = fare_Q3 + 1.5 * IQR

X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)

## 5. Building Preprocessing Pipelines

## 6. Label Encoding

## 7. Model Training

## 8. Evaluation